**Problem Overview:**

Consumers today rely heavily on online reviews when making purchase decisions. But with thousands of reviews scattered across platforms, it becomes difficult to identify the best products and most trustworthy feedback. This not only frustrates users but also hurts retailers who lose sales due to poor review navigation.
My project solves this challenge by using natural language processing and machine learning to clean, summarize, and extract insights from large volumes of product reviews.


In [1]:
!pip install -q dash xgboost pandas sentence-transformers faiss-cpu wordcloud
!apt -qq install -y nodejs npm
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 65.1 MB/s eta 0:00:00
The following additional packages will be inst

In [2]:
!python -m textblob.download_corpora
import textblob.download_corpora
textblob.download_corpora.download_all()

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
Finished.


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


In [3]:
import os
import dash
from dash import dcc, html, Input, Output, State
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import re, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, roc_auc_score
import xgboost as xgb
from sentence_transformers import SentenceTransformer
from textblob import TextBlob
import faiss
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import base64
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")

In [4]:
# === Load and preprocess dataset ===
df = pd.read_csv("amazon.csv")
df.drop_duplicates(inplace=True)
df.dropna(subset=['product_name', 'category'], inplace=True)
df['product_name'] = df['product_name'].str.lower().str.strip()
df['category'] = df['category'].str.lower().str.strip()
df['review_content'] = df['review_content'].fillna("")

# Drop empty columns
df.dropna(axis=1, how='all', inplace=True)

In [11]:
# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_review(text):
    text = re.sub(r'&lt;.*?&gt;', '', str(text))
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    words = text.split()
    return ' '.join([lemmatizer.lemmatize(w) for w in words if w not in stop_words])
df['cleaned_review'] = df['review_content'].apply(clean_review)
df['review_length'] = df['review_content'].apply(lambda x: len(x.split()))
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df.dropna(subset=['rating'])
df['rating'] = df['rating'].astype(float)
df['sentiment'] = df['review_content'].apply(lambda x: TextBlob(x).sentiment.polarity)
def safe_text(text):
    return text.encode('utf-16', 'surrogatepass').decode('utf-16', 'replace')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
# === Feature Engineering ===
tfidf = TfidfVectorizer(max_features=300)
tfidf_features = tfidf.fit_transform(df['cleaned_review']).toarray()
X_features = np.hstack((tfidf_features, df[['review_length']].values))
y_rating = df['rating']

y_helpful = np.random.randint(0, 2, size=len(df))

# Train/test split
X_train, X_test, y_train_rating, y_test_rating = train_test_split(X_features, y_rating, test_size=0.2, random_state=42)
_, _, y_train_helpful, y_test_helpful = train_test_split(X_features, y_helpful, test_size=0.2, random_state=42)

# === Model Training ===
model_rating = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
model_helpful = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, n_estimators=100, random_state=42)

model_rating.fit(X_train, y_train_rating)
model_helpful.fit(X_train, y_train_helpful)

# Predictions
y_pred_rating = model_rating.predict(X_test)
y_pred_helpful_proba = model_helpful.predict_proba(X_test)[:, 1]
y_pred_helpful = (y_pred_helpful_proba >= 0.5).astype(int)

# === Evaluation Metrics ===
rmse = np.sqrt(mean_squared_error(y_test_rating, y_pred_rating))
accuracy = accuracy_score(y_test_helpful, y_pred_helpful)
roc_auc = roc_auc_score(y_test_helpful, y_pred_helpful_proba)

# Update df with predictions for full dataset for other uses
df['pred_rating'] = model_rating.predict(X_features)
df['pred_helpful'] = model_helpful.predict_proba(X_features)[:, 1]
df['hybrid_score'] = df['sentiment'] * df['pred_rating'] * df['pred_helpful']

# Prepare top 2 products for A/B test
top_products = df.groupby("product_name").agg({
    "rating": "mean",
    "review_content": "count"
}).sort_values(by=["rating", "review_content"], ascending=False).head(2).index.tolist()

# === Other data prep (product stats, embeddings, etc.) ===
product_stats = df.groupby('product_name').agg(
    avg_rating=('rating', 'mean'),
    rating_count=('rating', 'count'),
    review_count=('review_content', 'count'),
    hybrid_score=('sentiment', lambda x: np.mean(x * df.loc[x.index, 'rating'] * df.loc[x.index, 'rating']))
).reset_index()


In [7]:
# Embeddings & FAISS
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

if 'cleaned_review' in df.columns:
    embeddings = model.encode(df['cleaned_review'].tolist(), show_progress_bar=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))
else:
    print("Error: 'cleaned_review' column not found in DataFrame. Please run the data cleaning cell first.")
    embeddings = None
    index = None


# Best/worst products
top_10_by_rating = product_stats.sort_values('avg_rating', ascending=False).head(10)
worst_10_by_rating = product_stats.sort_values('avg_rating', ascending=True).head(10)

# Summarization
def summarize_text(text):
    blob = TextBlob(text)
    sentences = blob.sentences
    if len(sentences) <= 2:
        return text
    return sentences[0].string + ' ' + sentences[-1].string

df['summary'] = df['review_content'].apply(summarize_text)

# Suggestion helper
def product_suggestion(product_name):
    subset = df[df['product_name'] == product_name]
    if subset.empty:
        return "No information available for this product."
    avg_score = subset['hybrid_score'].mean()
    avg_rating = subset['rating'].mean()
    sentiment_avg = subset['sentiment'].mean()
    helpful_avg = subset['pred_helpful'].mean()
    suggestion = (f"This product has an average hybrid score of {avg_score:.2f}, "
                  f"average rating of {avg_rating:.2f}, "
                  f"with generally {'positive' if sentiment_avg >= 0 else 'negative'} sentiment "
                  f"and helpfulness score of {helpful_avg:.2f}.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/46 [00:00<?, ?it/s]

In [14]:
# === Dash App ===
import datetime
ab_feedback_log = []
app = dash.Dash(__name__, suppress_callback_exceptions=True)
server = app.server

def compute_hybrid_score(df):
    try:
        if 'hybrid_score' not in df.columns:
            df['hybrid_score'] = (
                0.4 * df['rating'].astype(float) +
                0.4 * df['pred_helpful'].astype(float) +
                0.2 * df['sentiment'].astype(float)
            )
    except Exception as e:
        print(f"Error computing hybrid_score: {e}")
        df['hybrid_score'] = 0
    return df

def product_suggestion(product_name):
    try:
        name = product_name.lower()
        if "cable" in name:
            return "Consider bundling with a fast charger"
        elif "fan" in name:
            return "Good for summer season deals"
        elif "cover" in name:
            return "Great with screen protectors"
        return "Top performer in its category"
    except:
        return "No suggestion"

app.layout = html.Div([
    html.H1("📦 Product Review Dashboard", style={'textAlign': 'center'}),
    dcc.Tabs(id="tabs", value='tab-1', children=[
        dcc.Tab(label='🔍 Recommender', value='tab-1'),
        dcc.Tab(label='📊 Analytics', value='tab-2'),
        dcc.Tab(label='🧠 ML Insights', value='tab-3'),
    ]),
    html.Div(id='tabs-content')
])

@app.callback(Output('tabs-content', 'children'), Input('tabs', 'value'))
def render_content(tab):
    global df
    df = compute_hybrid_score(df)

    if tab == 'tab-1':
        return html.Div([
            html.H3("Semantic Search & Recommendations"),
            dcc.Input(id='search-box', type='text', placeholder='Search reviews...', style={'width': '50%'}),
            html.Br(), html.Br(),
            html.Label("Filter by Product:"),
            dcc.Dropdown(
                id='product-filter',
                options=[{'label': p, 'value': p} for p in sorted(df['product_name'].unique())],
                placeholder='Select product', style={'width': '50%'}),
            html.Div(id='search-results'),
            html.H4(" Top Recommendations Based on Hybrid Score"),
            html.Ul([
                html.Li(f"{row['product_name']} - Score: {row['hybrid_score']:.2f} - Suggestion: {product_suggestion(row['product_name'])}")
                for _, row in df.sort_values('hybrid_score', ascending=False).head(5).iterrows()
            ])
        ])

    elif tab == 'tab-2':
        wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(df['cleaned_review']))
        buffer = BytesIO()
        plt.figure(figsize=(10, 5))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(buffer, format='png')
        buffer.seek(0)
        img_base64 = base64.b64encode(buffer.read()).decode()

        return html.Div([
            dcc.Graph(figure=px.pie(df, names='rating', title="Rating Distribution")),
            dcc.Graph(figure=px.bar(df.groupby('rating')['sentiment'].mean().reset_index(),
                                    x='rating', y='sentiment', title="Avg Sentiment by Rating")),
            html.Img(src=f'data:image/png;base64,{img_base64}', style={'width': '100%'}),
            html.H4(" Top 10 Best Products by Average Rating"),
            dcc.Graph(figure=px.bar(top_10_by_rating, x='product_name', y='avg_rating', title="Top 10 Best Products by Avg Rating")),
            html.H4(" Top 10 Worst Products by Average Rating"),
            dcc.Graph(figure=px.bar(worst_10_by_rating, x='product_name', y='avg_rating', title="Top 10 Worst Products by Avg Rating")),
        ])
    elif tab == 'tab-3':
        return html.Div([
            html.H3("Model Evaluation Metrics"),
            html.P(f"Regression Model (Rating Prediction) RMSE: {rmse:.3f}"),
            html.P(f"Classification Model (Helpfulness Prediction) Accuracy: {accuracy:.3f}"),
            html.P(f"Classification Model ROC AUC: {roc_auc:.3f}"),
            html.Br(),
            html.H4("Feature Engineering Details"),
            html.Ul([
                html.Li("TF-IDF features: max 300 features from cleaned reviews"),
                html.Li("Additional features: review length"),
                html.Li("Models: XGBoost Regressor for rating, XGBoost Classifier for helpfulness"),
                html.Li("Random binary labels for helpfulness (replace with real data if available)"),
            ]),
            html.Br(),
            html.H4(" Top 5 Products by Hybrid Score"),
            html.Ul([
                html.Li(f"{row['product_name']} - Score: {row['hybrid_score']:.2f} - Suggestion: {product_suggestion(row['product_name'])}")
                for _, row in df.sort_values('hybrid_score', ascending=False).head(5).iterrows()
            ]),
            html.Hr(),
            html.H4("A/B Test: Compare Two Best Products"),
            html.Div(id='ab-product-selection', children=[
                html.Div([
                    html.H5(f"Product A: {top_products[0]}"),
                    dcc.Graph(id='ab-graph-a'),
                    html.Div(id='summary-a', style={'padding': '10px'})
                ], style={'width': '48%', 'display': 'inline-block'}),

                html.Div([
                    html.H5(f"Product B: {top_products[1]}"),
                    dcc.Graph(id='ab-graph-b'),
                    html.Div(id='summary-b', style={'padding': '10px'})
                ], style={'width': '48%', 'display': 'inline-block', 'float': 'right'}),
            ]),

            html.Div([
                html.H4(" Which product would you choose?"),
                dcc.RadioItems(
                    id='ab-feedback',
                    options=[
                        {'label': top_products[0], 'value': 'A'},
                        {'label': top_products[1], 'value': 'B'}
                    ],
                    labelStyle={'display': 'inline-block', 'margin-right': '20px'}
                ),
                html.Button("Submit Feedback", id='submit-ab', n_clicks=0),
                html.Div(id='feedback-confirmation', style={'margin-top': '10px', 'color': 'green'}),
                html.Hr(),
                html.Div(id='ab-comparison-summary')
            ])
        ])
    return html.Div([html.P("Tab content not found.")])

@app.callback(
    Output('ab-graph-a', 'figure'),
    Output('summary-a', 'children'),
    Output('ab-graph-b', 'figure'),
    Output('summary-b', 'children'),
    Input('tabs', 'value')
)
def update_ab_graphs(tab):
    if tab != 'tab-3':
        raise dash.exceptions.PreventUpdate


    prod_a = top_products[0]
    df_a = df[df['product_name'] == prod_a]
    fig_a = px.histogram(df_a, x='rating', nbins=5, title=f"{prod_a} - Rating Distribution")
    summary_a = summarize_text(' '.join(df_a['review_content'].astype(str).tolist()))


    prod_b = top_products[1]
    df_b = df[df['product_name'] == prod_b]
    fig_b = px.histogram(df_b, x='rating', nbins=5, title=f"{prod_b} - Rating Distribution")
    summary_b = summarize_text(' '.join(df_b['review_content'].astype(str).tolist()))

    return fig_a, summary_a, fig_b, summary_b


@app.callback(
    Output('search-results', 'children'),
    Input('search-box', 'value'),
    Input('product-filter', 'value')
)
def update_search_results(search_text, product_filter):
    filtered_df = df
    if product_filter:
        filtered_df = filtered_df[filtered_df['product_name'] == product_filter]

    if search_text and search_text.strip():

        q_vec = model.encode([search_text], convert_to_numpy=True)

        D, I = faiss_index.search(q_vec, 10)
        matched_indices = I[0]
        results = []
        for idx in matched_indices:
            if idx < len(filtered_df):
                row = df.iloc[idx]
                if product_filter and row['product_name'] != product_filter:
                    continue
                snippet = row['review_content'][:200] + ("..." if len(row['review_content']) > 200 else "")
                results.append(html.Div([
                    html.B(f"Product: {row['product_name']}"),
                    html.P(snippet),
                    html.Hr()
                ]))
        if not results:
            return "No results found."
        return results
    else:

        sample = filtered_df.sort_values('hybrid_score', ascending=False).head(5)
        return [html.Div([
            html.B(f"Product: {row['product_name']}"),
            html.P(row['review_content'][:200] + ("..." if len(row['review_content']) > 200 else "")),
            html.Hr()
        ]) for _, row in sample.iterrows()]
@app.callback(
    Output('feedback-confirmation', 'children'),
    Output('ab-comparison-summary', 'children'),
    Input('submit-ab', 'n_clicks'),
    State('ab-feedback', 'value')
)
def handle_ab_feedback(n_clicks, vote):
    global ab_feedback_log
    if n_clicks > 0 and vote:
        timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        ab_feedback_log.append({'timestamp': timestamp, 'vote': vote})

        vote_counts = pd.DataFrame(ab_feedback_log).vote.value_counts().to_dict()
        summary = [f"{top_products[0]}: {vote_counts.get('A', 0)} votes",
                   f"{top_products[1]}: {vote_counts.get('B', 0)} votes"]

        return f"Thank you for your feedback!", html.Ul([html.Li(s) for s in summary])
    return "", ""
if __name__ == '__main__':
    app.run(debug=True, port=8050)

<IPython.core.display.Javascript object>

In [15]:
import json

notebook_path = 'productreview_py.ipynb'

# Load the notebook
with open(notebook_path, 'r', encoding='utf-8') as f:
    nb = json.load(f)

widgets = nb.get('metadata', {}).get('widgets', {})
if "application/vnd.jupyter.widget-state+json" in widgets:
    widget_state = widgets["application/vnd.jupyter.widget-state+json"]
    if "state" not in widget_state:
        widget_state["state"] = {}

# Save the notebook
with open(notebook_path, 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1)

print("Fixed! The 'state' key has been added to metadata.widgets.")

FileNotFoundError: [Errno 2] No such file or directory: 'productreview_py.ipynb'